In [ ]:
import asyncio, time

# ====== MODEL COMPARISON ======
models = {
    "nomic-embed-text:v1.5": None,
    "qwen3-embedding:0.6b": None,   # Run first: ollama pull qwen3-embedding:0.6b
}

questions = [
    "What is the main architecture proposed in the paper?",
    "How does self-attention work in the Transformer model?",
    "What is multi-head attention and why is it used?",
    "What are the encoder and decoder components of the Transformer?",
    "What is positional encoding and why is it necessary?",
    "What BLEU score did the model achieve on English-to-German translation?",
    "How does the Transformer compare to recurrent neural networks?",
    "What is the scaled dot-product attention mechanism?",
    "How many parameters does the base Transformer model have?",
    "What training data and hardware were used for the experiments?",
]

all_scores = {}
for model_name in models:
    print(f"\n{'='*80}")
    print(f"Embedding with: {model_name}")
    print(f"{'='*80}")

    emb = OllamaEmbeddings(model=model_name)
    store = Chroma(
        collection_name=model_name.replace(":", "_"),
        embedding_function=emb,
        persist_directory=f"./chroma_compare_{model_name.replace(':', '_')}",
    )
    store.reset_collection()

    start = time.time()
    for i in range(0, len(all_splits), 10):
        store.add_documents(documents=all_splits[i:i + 10])
        print(f"  Embedded {min(i + 10, len(all_splits))}/{len(all_splits)} chunks...", end="\r")
    embed_time = time.time() - start
    print(f"\n  Done in {embed_time:.1f}s")

    async def run_queries(s, qs):
        tasks = [s.asimilarity_search_with_relevance_scores(q, k=1) for q in qs]
        return await asyncio.gather(*tasks)

    results = await run_queries(store, questions)
    scores = [r[0][1] for r in results]
    pages = [r[0][0].metadata.get("page_label", "?") for r in results]
    all_scores[model_name] = {"scores": scores, "pages": pages, "time": embed_time}

# ====== SIDE-BY-SIDE COMPARISON ======
model_names = list(all_scores.keys())
m1, m2 = model_names[0], model_names[1]

print(f"\n{'='*80}")
print(f"{'HEAD-TO-HEAD COMPARISON':^80}")
print(f"{'='*80}")
print(f"\n{'Question':<55} {'nomic':>8} {'qwen3':>8} {'Winner':>8}")
print("-" * 80)

wins = {m1: 0, m2: 0, "tie": 0}
for i, q in enumerate(questions):
    s1 = all_scores[m1]["scores"][i]
    s2 = all_scores[m2]["scores"][i]
    p1 = all_scores[m1]["pages"][i]
    p2 = all_scores[m2]["pages"][i]

    if s1 > s2 + 0.01:   winner = "nomic"; wins[m1] += 1
    elif s2 > s1 + 0.01: winner = "qwen3"; wins[m2] += 1
    else:                 winner = "tie";   wins["tie"] += 1

    tag = {"nomic": "◀", "qwen3": "▶", "tie": "="}[winner]
    print(f"{q[:53]:<55} {s1:>7.4f}  {s2:>7.4f}  {tag}")

avg1 = sum(all_scores[m1]["scores"]) / len(questions)
avg2 = sum(all_scores[m2]["scores"]) / len(questions)
t1 = all_scores[m1]["time"]
t2 = all_scores[m2]["time"]

print(f"\n{'='*80}")
print(f"{'SUMMARY':^80}")
print(f"{'='*80}")
print(f"  {'Metric':<25} {'nomic-embed-text:v1.5':>25} {'qwen3-embedding:0.6b':>25}")
print(f"  {'Avg Score':<25} {avg1:>25.4f} {avg2:>25.4f}")
print(f"  {'Embed Time':<25} {t1:>24.1f}s {t2:>24.1f}s")
print(f"  {'Wins':<25} {wins[m1]:>25} {wins[m2]:>25}")
print(f"\n  Winner: {'nomic-embed-text:v1.5 ◀' if avg1 > avg2 else 'qwen3-embedding:0.6b ▶'}")